# Building different tokenizer models from scratch


In [77]:
from tokenizers import Tokenizer, AddedToken, pre_tokenizers, processors, decoders
from tokenizers.models import BPE, WordPiece, Unigram
from tokenizers.trainers import BpeTrainer, WordPieceTrainer, UnigramTrainer
from tokenizers.pre_tokenizers import PreTokenizer, Whitespace
from tokenizers.normalizers import NFD, Lowercase, StripAccents, Sequence

In [78]:
# träningsfiler
files = ["svensk_text_1.txt",
        "svensk_text_2.txt",
        "svensk_text_3.txt",
        ]

# variables
vocab_size = 10000 
min_frequency = 4

In [79]:
text = "Det här är en exempelmening på svenska med åäö och sammansatta ord som e-post."
text = "Elmontörer kommer att spela en viktig roll i framtidens samhälle."
text = "Elmaterial har en viktig roll i framtidens samhälle att spela."
# text = "Byggmontör kommer att spela en viktig roll i framtidens samhälle."
# text = "Byggarbete kommer att spela en viktig roll i framtidens samhälle."
# text = "Byggmaterial kommer att spela en viktig roll i framtidens samhälle."
text = "Han behöver elmaterial för sitt elarbete som elmontör."
text

'Han behöver elmaterial för sitt elarbete som elmontör.'

### WordPiece model like BERT

In [80]:
# Build a tokenizer
bert_tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
bert_tokenizer.normalizer = Sequence([NFD(), Lowercase()])
bert_tokenizer.pre_tokenizer = Whitespace()

# Define special tokens
special_tokens = [
    "[PAD]",   # Padding token
    "[UNK]",   # Unknown token
    "[CLS]",   # Classification token
    "[SEP]",   # Separator token
    "[MASK]",  # Masking token
]

# Initilize trainer
trainer = WordPieceTrainer(
    vocab_size=vocab_size,
    min_frequency=min_frequency,
    special_tokens=special_tokens,
    continuing_subword_prefix="##",
)

# Train the tokenizer on the provided files
bert_tokenizer.train(files, trainer)

# Save the trained tokenizer to a JSON file
bert_output_path = "bert-custom-tokenizer.json"
bert_tokenizer.save(bert_output_path)

# Print confirmation and vocabulary size
print(f"Tokenizer trained and saved to {bert_output_path}")
print(f"Vocab size after training: {bert_tokenizer.get_vocab_size()}")




Tokenizer trained and saved to bert-custom-tokenizer.json
Vocab size after training: 2516


### BPE model like GPT-2

In [81]:
# Initialize the tokenizer model
bpe_tokenizer = Tokenizer(BPE())
bpe_tokenizer.normalizer = Sequence([NFD(), Lowercase()])
bpe_tokenizer.pre_tokenizer = Whitespace()

# Define special tokens
special_tokens = ["<|endoftext|>"]

# Initialize the trainer
trainer = BpeTrainer(
    vocab_size=vocab_size,
    min_frequency=min_frequency,
    special_tokens=special_tokens,
)

# Train the tokenizer on the provided files
bpe_tokenizer.train(files, trainer)

# Save the trained tokenizer to a JSON file
bpe_output_path = "bpe-custom-tokenizer.json"
bpe_tokenizer.save(bpe_output_path)

# Print confirmation and vocabulary size
print(f"Tokenizer trained and saved to {bpe_output_path}")
print(f"Vocab size after training: {bpe_tokenizer.get_vocab_size()}")




Tokenizer trained and saved to bpe-custom-tokenizer.json
Vocab size after training: 2221


In [82]:
# Initialize the tokenizer model
bpeb_tokenizer = Tokenizer(BPE())
bpeb_tokenizer.normalizer = Sequence([NFD(), Lowercase()])
# change pre_tokenizers
bpeb_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)


# Define special tokens
special_tokens = ["<|endoftext|>"]

# Initialize the trainer
trainer = BpeTrainer(
    vocab_size=vocab_size,
    min_frequency=min_frequency,
    special_tokens=special_tokens,
)

# Train the tokenizer on the provided files
bpeb_tokenizer.train(files, trainer)

# ByteLevel need to include the post-processor and decoder
bpeb_tokenizer.post_processor = processors.ByteLevel(trim_offsets=True)
bpeb_tokenizer.decoder = decoders.ByteLevel()

# Save the trained tokenizer to a JSON file
bpeb_output_path = "bpeb-custom-tokenizer.json"
bpeb_tokenizer.save(bpeb_output_path)

# Print confirmation and vocabulary size
print(f"Tokenizer trained and saved to {bpeb_output_path}")
print(f"Vocab size after training: {bpeb_tokenizer.get_vocab_size()}")




Tokenizer trained and saved to bpeb-custom-tokenizer.json
Vocab size after training: 2308


### Unigram model like Albert

In [83]:
uni_tokenizer = Tokenizer(Unigram())
uni_tokenizer.normalizer = Sequence([NFD(), Lowercase()])
uni_tokenizer.pre_tokenizer = Whitespace()

# Specialtoken är desamma
special_tokens = [
    "[PAD]",
    "[UNK]",
    "[CLS]",
    "[SEP]",
    "[MASK]",
]

# Initiera UnigramTrainer
trainer = UnigramTrainer(
    vocab_size=vocab_size,
    special_tokens=special_tokens,
    # Unigram behöver veta vilken token som är UNK
    unk_token="[UNK]",
)


uni_tokenizer.train(files, trainer)

# --- 7. Spara tokenizern ---
# Sparar konfigurationen och den tränade vokabulären till en JSON-fil.
uni_output_path = "uni-custom-tokenizer.json"
uni_tokenizer.save(uni_output_path)

print(f"Tokenizer tränad och sparad till {uni_output_path}")
print(f"Vocab size after training: {uni_tokenizer.get_vocab_size()}")



Tokenizer tränad och sparad till uni-custom-tokenizer.json
Vocab size after training: 2422


## Load tokenizer

In [84]:
# Restoring model from learned config/vocab
loaded_bert_tokenizer = Tokenizer.from_file(bert_output_path)
loaded_bpe_tokenizer = Tokenizer.from_file(bpe_output_path)
loaded_bpeb_tokenizer = Tokenizer.from_file(bpeb_output_path)
loaded_uni_tokenizer = Tokenizer.from_file(uni_output_path)

# Test encoding
bert_encode = loaded_bert_tokenizer.encode(text)
bpe_encode = loaded_bpe_tokenizer.encode(text)
bpeb_encode = loaded_bpeb_tokenizer.encode(text)
uni_encode = loaded_uni_tokenizer.encode(text)

# token information
print("Originaltext:", text)
print("BERT Tokens:", bert_encode.tokens)
print("BPE Tokens (space):", bpe_encode.tokens)
print("BPE Tokens (bytes):", bpeb_encode.tokens)
print("Unicode Tokens:", uni_encode.tokens)

Originaltext: Han behöver elmaterial för sitt elarbete som elmontör.
BERT Tokens: ['ha', '##n', 'behöver', 'elmaterial', 'för', 'sitt', 'elarbete', 'som', 'elmontör', '.']
BPE Tokens (space): ['han', 'behöver', 'elmaterial', 'för', 'sitt', 'elarbete', 'som', 'elmontör', '.']
BPE Tokens (bytes): ['han', 'Ġbeho', 'ÌĪ', 'ver', 'Ġelmaterial', 'Ġfo', 'ÌĪ', 'r', 'Ġsitt', 'Ġelarbete', 'Ġsom', 'Ġelmonto', 'ÌĪ', 'r', '.']
Unicode Tokens: ['ha', 'n', 'behöv', 'er', 'elmaterial', 'för', 'sitt', 'elarbete', 'som', 'elmontör', '.']


In [85]:
# decoded text
print("Originaltext............:", text)
print("BERT decoded text.......:", loaded_bert_tokenizer.decode(bert_encode.ids))
print("BPE (space) decoded text:", loaded_bpe_tokenizer.decode(bpe_encode.ids))
print("BPE (bytes) decoded text:", loaded_bpeb_tokenizer.decode(bpeb_encode.ids))
print("Unicode decoded text....:", loaded_uni_tokenizer.decode(uni_encode.ids))

Originaltext............: Han behöver elmaterial för sitt elarbete som elmontör.
BERT decoded text.......: ha ##n behöver elmaterial för sitt elarbete som elmontör .
BPE (space) decoded text: han behöver elmaterial för sitt elarbete som elmontör .
BPE (bytes) decoded text: han behöver elmaterial för sitt elarbete som elmontör.
Unicode decoded text....: ha n behöv er elmaterial för sitt elarbete som elmontör .
